In [1]:
!pip install langgraph langchain-groq langchain-core google-search-results ddgs rich python-dotenv -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.2 MB/s eta 0:00:00:00:01


In [2]:
import os

def _load_keys():
    groq_key = serpapi_key = ""
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        groq_key    = s.get_secret("GROQ_API_KEY")
        serpapi_key = s.get_secret("SERPAPI_KEY")
        print("✅ Keys loaded from Kaggle Secrets")
        return groq_key, serpapi_key
    except Exception:
        pass

    groq_key    = os.getenv("GROQ_API_KEY", "")
    serpapi_key = os.getenv("SERPAPI_KEY", "")
    if groq_key and serpapi_key:
        print("Keys loaded from environment variables")
        return groq_key, serpapi_key

    if not groq_key:
        groq_key    = input("🔑 Enter GROQ_API_KEY: ").strip()
    if not serpapi_key:
        serpapi_key = input("🔑 Enter SERPAPI_KEY:  ").strip()

    print("✅ Keys ready")
    return groq_key, serpapi_key

GROQ_API_KEY, SERPAPI_KEY = _load_keys()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print(f"   Groq   : ...{GROQ_API_KEY[-6:]}")
print(f"   SerpAPI: ...{SERPAPI_KEY[-6:]}")

✅ Keys loaded from Kaggle Secrets
   Groq   : ...uLe7q4
   SerpAPI: ...ece889


In [3]:
import os, warnings

warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")
warnings.filterwarnings("ignore", category=ResourceWarning)

MODEL_NAME       = os.getenv("GROQ_MODEL",    "llama-3.3-70b-versatile")
CURRENCY         = os.getenv("TRAVEL_CURRENCY","USD")
LANGUAGE         = os.getenv("TRAVEL_LANG",   "en")
PROVIDER_TIMEOUT = int(os.getenv("SEARCH_TIMEOUT", "20"))
MAX_HISTORY      = int(os.getenv("MAX_HISTORY",     "10"))

LITE_MODE = os.getenv("LITE_MODE", "false").lower() == "true"

FORCE_FAIL_SEARCHES: set = set()

SEARCH_CONFIGS = [
    {"sort_by": "1", "label": "best",     "emoji": "🏆"},
    {"sort_by": "2", "label": "cheapest", "emoji": "💰"},
    {"sort_by": "5", "label": "fastest",  "emoji": "⚡"},
]

print(f"Config | model={MODEL_NAME} | lite={LITE_MODE} | "
      f"timeout={PROVIDER_TIMEOUT}s | currency={CURRENCY}")

Config | model=llama-3.3-70b-versatile | lite=False | timeout=20s | currency=USD


In [4]:
import re, json, time, random, warnings
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, List, Optional, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FTE
import operator

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from ddgs import DDGS                          # ← fixed: was duckduckgo_search
from serpapi import GoogleSearch
from rich.console import Console
from rich.panel   import Panel
from rich.table   import Table
from rich         import box

console = Console()

In [5]:
class TravelState(TypedDict):
    user_input:            str
    conversation_history:  List[Dict[str, str]]
    parsed_data:           Optional[Dict[str, Any]]
    parse_error:           Optional[str]
    origin_iata:           Optional[str]
    destination_iata:      Optional[str]
    origin_display:        Optional[str]
    destination_display:   Optional[str]
    airport_resolution_notes: Optional[str]
    validation_result:     Optional[Dict[str, Any]]
    needs_clarification:   bool
    clarification_message: Optional[str]
    flight_results:        Annotated[List[Dict], operator.add]
    total_searches_attempted:  int
    total_searches_succeeded:  int
    best_flight:           Optional[Dict]
    cheapest_flight:       Optional[Dict]
    fastest_flight:        Optional[Dict]
    all_flights_ranked:    Optional[List[Dict]]
    price_insights:        Optional[Dict]
    final_response:        Optional[str]
    execution_trace:       Annotated[List[Dict], operator.add]


def make_state(user_input: str, history: List = None) -> TravelState:
    return TravelState(
        user_input=user_input,
        conversation_history=history or [],
        parsed_data=None,          parse_error=None,
        origin_iata=None,          destination_iata=None,
        origin_display=None,       destination_display=None,
        airport_resolution_notes=None,
        validation_result=None,    needs_clarification=False,
        clarification_message=None,
        flight_results=[],         total_searches_attempted=0,
        total_searches_succeeded=0,
        best_flight=None,          cheapest_flight=None,
        fastest_flight=None,       all_flights_ranked=None,
        price_insights=None,
        final_response=None,       execution_trace=[],
    )


In [6]:
def get_llm(temperature: float = 0.0) -> ChatGroq:
    return ChatGroq(
        model=MODEL_NAME, temperature=temperature,
        api_key=GROQ_API_KEY, max_tokens=1024
    )

def safe_json(text: str) -> Optional[Dict]:
    for pat in [r'```json\s*([\s\S]*?)```', r'```\s*([\s\S]*?)```', r'(\{[\s\S]*\})']:
        m = re.search(pat, text)
        if m:
            try: return json.loads(m.group(1).strip())
            except: pass
    try: return json.loads(text.strip())
    except: return None

def resolve_relative_date(raw: str) -> Optional[str]:
    raw   = raw.lower().strip()
    today = datetime.now()
    simple = {
        "today":0,"tomorrow":1,"day after tomorrow":2,
        "next week":7,"next month":30,"this weekend":5,"next weekend":6,
    }
    for phrase, delta in simple.items():
        if phrase in raw:
            return (today + timedelta(days=delta)).strftime("%Y-%m-%d")
    weekdays = {"monday":0,"tuesday":1,"wednesday":2,"thursday":3,
                "friday":4,"saturday":5,"sunday":6}
    for name, idx in weekdays.items():
        if name in raw:
            ahead = idx - today.weekday()
            if ahead <= 0: ahead += 7
            return (today + timedelta(days=ahead)).strftime("%Y-%m-%d")
    m = re.search(r'(\d{4}-\d{2}-\d{2})', raw)
    if m: return m.group(1)
    return None

def trace_entry(agent: str, t0: float, inp: str, out: str,
                status: str = "success") -> Dict:
    return {
        "agent":          agent,
        "status":         status,
        "duration_ms":    round((time.time() - t0) * 1000, 1),
        "input_summary":  (inp or "")[:80],
        "output_summary": (out or "")[:80],
        "timestamp":      datetime.now().isoformat(),
    }


In [7]:
import time

_KNOWN_CITIES: Dict[str, str] = {
    # North America
    "new york":       "JFK",
    "new york city":  "JFK",
    "nyc":            "JFK",
    "los angeles":    "LAX",
    "la":             "LAX",
    "chicago":        "ORD",
    "miami":          "MIA",
    "san francisco":  "SFO",
    "seattle":        "SEA",
    "boston":         "BOS",
    "washington":     "IAD",
    "washington dc":  "IAD",
    "dallas":         "DFW",
    "houston":        "IAH",
    "atlanta":        "ATL",
    "denver":         "DEN",
    "phoenix":        "PHX",
    "las vegas":      "LAS",
    "orlando":        "MCO",
    "toronto":        "YYZ",
    "montreal":       "YUL",
    "vancouver":      "YVR",
    "mexico city":    "MEX",
    # Europe
    "london":         "LHR",
    "paris":          "CDG",
    "amsterdam":      "AMS",
    "frankfurt":      "FRA",
    "madrid":         "MAD",
    "barcelona":      "BCN",
    "rome":           "FCO",
    "milan":          "MXP",
    "zurich":         "ZRH",
    "vienna":         "VIE",
    "brussels":       "BRU",
    "lisbon":         "LIS",
    "munich":         "MUC",
    "berlin":         "BER",
    "copenhagen":     "CPH",
    "stockholm":      "ARN",
    "oslo":           "OSL",
    "helsinki":       "HEL",
    "athens":         "ATH",
    "istanbul":       "IST",
    # Asia Pacific
    "tokyo":          "NRT",
    "osaka":          "KIX",
    "seoul":          "ICN",
    "beijing":        "PEK",
    "shanghai":       "PVG",
    "hong kong":      "HKG",
    "singapore":      "SIN",
    "bangkok":        "BKK",
    "kuala lumpur":   "KUL",
    "jakarta":        "CGK",
    "delhi":          "DEL",
    "new delhi":      "DEL",
    "mumbai":         "BOM",
    "bangalore":      "BLR",
    "sydney":         "SYD",
    "melbourne":      "MEL",
    "auckland":       "AKL",
    # Middle East & Africa
    "dubai":          "DXB",
    "abu dhabi":      "AUH",
    "doha":           "DOH",
    "riyadh":         "RUH",
    "cairo":          "CAI",
    "johannesburg":   "JNB",
    "nairobi":        "NBO",
    # South America
    "sao paulo":      "GRU",
    "buenos aires":   "EZE",
}

_IATA_BLOCKLIST = {
    "THE","AND","FOR","BUT","NOT","ARE","WAS","HAS","HAD","ITS",
    "ALL","NEW","ONE","TWO","ANY","CAN","MAY","GET","SET","USE",
    "URL","API","CSS","USA","USD","EUR","GBP","GMT","UTC","EST",
    "PDT","PST","NYC",
}

def _ddg_search(query: str, max_results: int = 5) -> List[Dict]:
    for attempt in range(3):
        try:
            with DDGS() as ddgs:
                return list(ddgs.text(query, max_results=max_results))
        except Exception as e:
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    return []

def _extract_iata_from_text(text: str, city: str) -> Optional[str]:
    m = re.search(r'IATA\s*(?:code)?\s*[:\s]+([A-Z]{3})\b', text, re.IGNORECASE)
    if m:
        code = m.group(1).upper()
        if code not in _IATA_BLOCKLIST:
            return code

    for m in re.finditer(r'\(([A-Z]{3})\)', text):
        code = m.group(1)
        if code not in _IATA_BLOCKLIST:
            return code

    for m in re.finditer(r'\b([A-Z]{3})\s+(?:Airport|International|Intl)\b',
                         text, re.IGNORECASE):
        code = m.group(1).upper()
        if code not in _IATA_BLOCKLIST:
            return code

    m = re.search(r'code\s+([A-Z]{3})\b', text, re.IGNORECASE)
    if m:
        code = m.group(1).upper()
        if code not in _IATA_BLOCKLIST:
            return code

    return None

def lookup_iata(city: str) -> Dict[str, str]:
    """
    Returns {"iata": "JFK", "display": "New York (JFK)"}
    Priority: (1) already IATA, (2) known-cities dict, (3) DuckDuckGo, (4) city fallback
    """
    city = city.strip()

    if len(city) == 3 and city.isupper():
        return {"iata": city, "display": city}

    known_iata = _KNOWN_CITIES.get(city.lower())
    if known_iata:
        console.print(f"  ✅ IATA fast-path: [yellow]{city}[/yellow] → [green]{known_iata}[/green]")
        return {"iata": known_iata, "display": f"{city.title()} ({known_iata})"}

    query   = f"{city} main international airport IATA code"
    results = _ddg_search(query, max_results=5)

    if not results:
        console.print(f"  [yellow]⚠️  No DDG results for '{city}', using city name[/yellow]")
        return {"iata": city, "display": city}

    combined = " ".join(
        r.get("title","") + " " + r.get("body","")
        for r in results[:4]
    )

    iata = _extract_iata_from_text(combined, city)

    if iata:
        display_match = re.search(
            rf'{re.escape(city)}[^.]*?([A-Za-z\s]+(?:International|Airport)[^,.(]+)',
            combined, re.IGNORECASE
        )
        display = (f"{display_match.group(1).strip()} ({iata})"
                   if display_match else f"{city} ({iata})")
        return {"iata": iata, "display": display}

    console.print(f"  [yellow]⚠️  IATA not found for '{city}', using city name directly[/yellow]")
    return {"iata": city, "display": city}


In [8]:
import time, re
from typing import Dict, List, Optional

_iata_cache: Dict[str, Dict[str, str]] = {}

_IATA_BLOCKLIST = {
    "THE","AND","FOR","BUT","NOT","ARE","WAS","HAS","HAD","ITS",
    "ALL","NEW","ONE","TWO","ANY","CAN","MAY","GET","SET","USE",
    "URL","API","CSS","USA","USD","EUR","GBP","GMT","UTC","EST",
    "PDT","PST","NYC",
}



def _ddg_search(query: str, max_results: int = 5) -> List[Dict]:
    """DuckDuckGo text search with retry."""
    for attempt in range(3):
        try:
            with DDGS() as ddgs:
                return list(ddgs.text(query, max_results=max_results))
        except Exception:
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    return []


def _extract_iata_from_text(text: str) -> Optional[str]:
    """Regex extraction of IATA code from a block of text."""
    patterns = [
        r'IATA\s*(?:code)?\s*[:\s]+([A-Z]{3})\b',
        r'\(([A-Z]{3})\)',
        r'\b([A-Z]{3})\s+(?:Airport|International|Intl)\b',
        r'code\s+([A-Z]{3})\b',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            code = m.group(1).upper()
            if code not in _IATA_BLOCKLIST and len(code) == 3:
                return code
    return None


def _llm_lookup_iata(city: str) -> Optional[str]:
    """
    Ask the LLM for the IATA code.
    Groq's LLaMA model has every major and minor airport in its training data.
    Returns a validated 3-letter code or None.
    """
    try:
        from langchain_groq import ChatGroq
        from langchain_core.prompts import ChatPromptTemplate

        llm = ChatGroq(
            model=MODEL_NAME,
            temperature=0.0,
            api_key=GROQ_API_KEY,
            max_tokens=10,          # we only need 3 characters
        )
        chain = ChatPromptTemplate.from_messages([
            ("system",
             "You are an airport code expert. "
             "Given any city, region, or airport name, return ONLY the primary "
             "international airport IATA code — exactly 3 uppercase letters, nothing else. "
             "No punctuation, no explanation, no markdown. "
             "If genuinely unknown, return the 3 letters UNK."),
            ("human", "{city}")
        ]) | llm

        resp = chain.invoke({"city": city})
        code = resp.content.strip().upper()

        # Validate: exactly 3 uppercase letters and not our placeholder
        if re.match(r'^[A-Z]{3}$', code) and code != "UNK" and code not in _IATA_BLOCKLIST:
            return code
        return None

    except Exception as e:
        console.print(f"  [dim yellow]LLM IATA lookup failed for '{city}': {e}[/dim yellow]")
        return None


def _ddg_lookup_iata(city: str) -> Optional[str]:
    """DuckDuckGo fallback for cities the LLM couldn't confidently resolve."""
    results = _ddg_search(f"{city} international airport IATA code", max_results=5)
    if not results:
        return None
    combined = " ".join(
        r.get("title", "") + " " + r.get("body", "")
        for r in results[:4]
    )
    return _extract_iata_from_text(combined)


def _build_display(city: str, iata: str) -> str:
    """Human-readable display string e.g. 'Guwahati (GAU)'."""
    return f"{city.strip().title()} ({iata})"



def lookup_iata(city: str) -> Dict[str, str]:
    """
    Resolve any city/airport name → IATA code.

    Returns:
        {"iata": "GAU", "display": "Guwahati (GAU)"}

    Falls back gracefully at every step — never crashes the pipeline.
    """
    city = city.strip()
    if not city:
        return {"iata": "", "display": ""}

    if re.match(r'^[A-Z]{3}$', city):
        return {"iata": city, "display": city}

    cache_key = city.lower()

    if cache_key in _iata_cache:
        cached = _iata_cache[cache_key]
        console.print(
            f"  [dim]💾 cache hit: [yellow]{city}[/yellow] → "
            f"[green]{cached['iata']}[/green][/dim]"
        )
        return cached

    console.print(f"  🔍 resolving [yellow]{city}[/yellow] via LLM…")
    iata = _llm_lookup_iata(city)
    source = "LLM"

    if not iata:
        console.print(f"  [dim]LLM miss → trying DuckDuckGo…[/dim]")
        iata = _ddg_lookup_iata(city)
        source = "DuckDuckGo"

    if not iata:
        console.print(
            f"  [yellow]⚠️  Could not resolve IATA for '{city}'. "
            f"Passing city name to SerpAPI — it may still work.[/yellow]"
        )
        result = {"iata": city, "display": city}
        _iata_cache[cache_key] = result
        return result

    display = _build_display(city, iata)
    console.print(
        f"  ✅ [yellow]{city}[/yellow] → [green]{iata}[/green] "
        f"[dim]({source})[/dim]"
    )

    result = {"iata": iata, "display": display}
    _iata_cache[cache_key] = result
    return result


In [9]:
_PARSER_SYSTEM = """You are a travel request parser.
Return ONLY a JSON object — no markdown, no extra text.
Keys:
  destination    : string or null  (city name e.g. "Paris")
  origin         : string (default "New York" if not stated)
  raw_date_text  : exact phrase user said for date, e.g. "next Friday", or null
  travel_date    : YYYY-MM-DD if date is explicit and absolute, else null
  travel_type    : flight|hotel|train|cruise or null
  num_passengers : integer (default 1)
  cabin_class    : economy|business|first|premium_economy (default "economy")
  budget_hint    : string or null (e.g. "cheap","under $500","business class")
Today is {today}."""

def parser_agent(state: TravelState) -> Dict:
    t0 = time.time()
    console.print("\n[bold blue]🔤 ParserAgent[/bold blue]")

    history_ctx = "".join(
        f"\n{m['role']}: {m['content']}"
        for m in state.get("conversation_history",[])[-4:]
    )

    try:
        llm   = get_llm(temperature=0.0)
        chain = ChatPromptTemplate.from_messages([
            ("system", _PARSER_SYSTEM.format(today=datetime.now().strftime("%Y-%m-%d"))),
            ("human", "Request: {inp}\nConversation history:{hist}")
        ]) | llm

        resp   = chain.invoke({"inp": state["user_input"], "hist": history_ctx})
        parsed = safe_json(resp.content)

        if not parsed:
            raise ValueError(f"No JSON in LLM output: {resp.content[:120]}")

        if not parsed.get("travel_date") and parsed.get("raw_date_text"):
            parsed["travel_date"] = resolve_relative_date(parsed["raw_date_text"])

        console.print(
            f"  ✅ dest=[yellow]{parsed.get('destination')}[/yellow] "
            f"origin=[yellow]{parsed.get('origin')}[/yellow] "
            f"date=[yellow]{parsed.get('travel_date')}[/yellow] "
            f"pax=[yellow]{parsed.get('num_passengers')}[/yellow] "
            f"cabin=[yellow]{parsed.get('cabin_class')}[/yellow]"
        )

        return {
            "parsed_data": parsed, "parse_error": None,
            "execution_trace": [trace_entry(
                "ParserAgent", t0, state["user_input"][:60],
                f"dest={parsed.get('destination')}, date={parsed.get('travel_date')}"
            )]
        }
    except Exception as e:
        console.print(f"  ❌ {e}")
        return {
            "parsed_data": None, "parse_error": str(e),
            "execution_trace": [trace_entry("ParserAgent", t0,
                state["user_input"][:60], str(e), "error")]
        }


In [10]:


def airport_resolver_agent(state: TravelState) -> Dict:
    t0 = time.time()
    console.print("\n[bold blue]🗺️  AirportResolverAgent[/bold blue]")

    p = state.get("parsed_data") or {}
    origin_city = p.get("origin", "New York")
    dest_city   = p.get("destination", "")

    if not dest_city:
        return {
            "origin_iata": origin_city, "destination_iata": dest_city or "",
            "origin_display": origin_city, "destination_display": dest_city or "",
            "airport_resolution_notes": "no destination to resolve",
            "execution_trace": [trace_entry("AirportResolverAgent", t0,
                "no destination", "skipped", "skipped")]
        }

    results = {}
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {
            pool.submit(lookup_iata, origin_city): "origin",
            pool.submit(lookup_iata, dest_city):   "destination",
        }
        for fut in as_completed(futures, timeout=15):
            key = futures[fut]
            try:
                results[key] = fut.result(timeout=12)
            except Exception as exc:
                city = origin_city if key == "origin" else dest_city
                results[key] = {"iata": city, "display": city}
                console.print(f"  ⚠️  {key} lookup failed: {exc}")

    origin_res = results.get("origin", {"iata": origin_city, "display": origin_city})
    dest_res   = results.get("destination", {"iata": dest_city, "display": dest_city})

    # ── Safety net: if iata is still a multi-word city name, warn ─────────
    for label, res in [("origin", origin_res), ("destination", dest_res)]:
        if " " in res["iata"]:
            console.print(
                f"  [red]⚠️  {label} IATA resolved to '{res['iata']}' "
                f"(multi-word — SerpAPI may reject). "
                f"Add this city to _KNOWN_CITIES.[/red]"
            )

    console.print(
        f"  ✅ origin=[yellow]{origin_res['iata']}[/yellow]"
        f" ({origin_res['display']}) | "
        f"dest=[yellow]{dest_res['iata']}[/yellow]"
        f" ({dest_res['display']})"
    )

    ms = round((time.time()-t0)*1000)
    return {
        "origin_iata":              origin_res["iata"],
        "destination_iata":         dest_res["iata"],
        "origin_display":           origin_res["display"],
        "destination_display":      dest_res["display"],
        "airport_resolution_notes": f"resolved in {ms}ms",
        "execution_trace": [trace_entry(
            "AirportResolverAgent", t0,
            f"{origin_city} + {dest_city}",
            f"{origin_res['iata']} + {dest_res['iata']}"
        )]
    }


In [11]:
_REQUIRED = {
    "flight":  ["destination", "travel_date"],
    "hotel":   ["destination", "travel_date"],
    "train":   ["destination", "travel_date"],
    "cruise":  ["destination", "travel_date"],
    "default": ["destination"],
}
_CLARIFY = {
    "destination": "Where would you like to travel? (city name)",
    "travel_date": "When do you want to travel? (e.g. 'next Friday' or 'Dec 15')",
    "travel_type": "What type of travel? (flight / hotel / train / cruise)",
}

def validator_agent(state: TravelState) -> Dict:
    t0 = time.time()
    console.print("\n[bold blue]✔️  ValidatorAgent[/bold blue]")

    if state.get("parse_error") or not state.get("parsed_data"):
        msg = ("I couldn't parse that request. "
               "Please try: 'Book a flight to Paris on December 15th'")
        return {
            "validation_result":   {"is_valid": False, "errors": ["parse_failed"]},
            "needs_clarification": True,
            "clarification_message": msg,
            "execution_trace": [trace_entry("ValidatorAgent", t0,
                "no data", msg[:60], "clarification_needed")]
        }

    p      = state["parsed_data"]
    ttype  = (p.get("travel_type") or "default").lower()
    req    = _REQUIRED.get(ttype, _REQUIRED["default"])
    missing, errors = [], []

    for f in req:
        if not p.get(f):
            missing.append(f)

    if p.get("travel_date"):
        try:
            td = datetime.strptime(p["travel_date"], "%Y-%m-%d")
            if td.date() < datetime.now().date():
                errors.append(f"Date {p['travel_date']} is in the past")
        except ValueError:
            errors.append(f"Unrecognised date format: {p['travel_date']}")
            missing.append("travel_date")

    is_valid    = not missing and not errors
    clarify_msg = None
    if not is_valid:
        parts = errors + [_CLARIFY.get(f, f"Please provide {f}") for f in missing]
        clarify_msg = "I need a bit more info: " + "  ⬤  ".join(parts)

    status = "success" if is_valid else "clarification_needed"
    console.print(f"  {'✅ Valid' if is_valid else f'⚠️  missing={missing} errors={errors}'}")

    return {
        "validation_result": {
            "is_valid": is_valid, "missing_fields": missing,
            "errors": errors, "travel_type": ttype,
        },
        "needs_clarification":   not is_valid,
        "clarification_message": clarify_msg,
        "execution_trace": [trace_entry("ValidatorAgent", t0,
            f"required={req}", f"valid={is_valid}, missing={missing}", status)]
    }

print("ValidatorAgent ready")

ValidatorAgent ready


In [12]:
def _parse_serpapi_flights(data: Dict, sort_label: str) -> List[Dict]:
    flights = []
    for category in ["best_flights", "other_flights"]:
        for item in data.get(category, [])[:5]:
            segs = item.get("flights", [])
            if not segs or not item.get("price"):
                continue
            first, last = segs[0], segs[-1]
            dep_airport = first.get("departure_airport", {})
            arr_airport = last.get("arrival_airport", {})
            layovers    = item.get("layovers", [])
            flights.append({
                "airline":          first.get("airline", "Unknown"),
                "airline_logo":     first.get("airline_logo", ""),
                "flight_number":    first.get("flight_number", ""),
                "price_usd":        item.get("price", 0),
                "duration_min":     item.get("total_duration", 0),
                "duration_h":       round(item.get("total_duration", 0) / 60, 1),
                "departure_time":   dep_airport.get("time", ""),
                "departure_code":   dep_airport.get("id", ""),
                "arrival_time":     arr_airport.get("time", ""),
                "arrival_code":     arr_airport.get("id", ""),
                "stops":            len(layovers),
                "stops_label":      "Direct" if not layovers else f"{len(layovers)} stop(s)",
                "layover_info":     [l.get("name","") for l in layovers],
                "cabin_class":      first.get("travel_class", "Economy"),
                "airplane":         first.get("airplane", ""),
                "legroom":          first.get("legroom", ""),
                "category":         category.replace("_", " ").title(),
                "sort_perspective": sort_label,
                "num_segments":     len(segs),
            })
    return flights


def _fetch_one_search(sort_label, sort_by, departure_iata, arrival_iata,
                      travel_date, num_passengers, cabin_class) -> Dict:
    cabin_map = {
        "economy":"1", "premium_economy":"2", "business":"3", "first":"4",
    }
    cabin_code = cabin_map.get(cabin_class.lower(), "1")

    params = {
        "engine":         "google_flights",
        "departure_id":   departure_iata,
        "arrival_id":     arrival_iata,
        "outbound_date":  travel_date,
        "currency":       CURRENCY,
        "hl":             LANGUAGE,
        "adults":         str(num_passengers),
        "type":           "2",
        "sort_by":        sort_by,
        "travel_class":   cabin_code,
        "api_key":        SERPAPI_KEY,
    }

    if sort_label in FORCE_FAIL_SEARCHES:
        raise RuntimeError(f"[test] forced failure for '{sort_label}' search")

    search = GoogleSearch(params)
    data   = search.get_dict()

    if "error" in data:
        raise RuntimeError(f"SerpAPI error: {data['error']}")
    status = data.get("search_metadata", {}).get("status", "")
    if status != "Success":
        raise RuntimeError(f"SerpAPI status: {status}")

    return {
        "label":          sort_label,
        "flights":        _parse_serpapi_flights(data, sort_label),
        "price_insights": data.get("price_insights", {}),
        "total_found":    len(_parse_serpapi_flights(data, sort_label)),
    }


In [13]:
def _run_search(config: Dict, state: TravelState) -> Dict:
    p     = state["parsed_data"] or {}
    label = config["label"]
    t0    = time.time()
    try:
        result = _fetch_one_search(
            sort_label      = label,
            sort_by         = config["sort_by"],
            departure_iata  = state.get("origin_iata") or p.get("origin","JFK"),
            arrival_iata    = state.get("destination_iata") or p.get("destination",""),
            travel_date     = p.get("travel_date", datetime.now().strftime("%Y-%m-%d")),
            num_passengers  = p.get("num_passengers", 1),
            cabin_class     = p.get("cabin_class", "economy"),
        )
        ms = round((time.time()-t0)*1000)
        console.print(
            f"  ✅ {config['emoji']} [green]{label:10}[/green] "
            f"→ {result['total_found']} flights found  ({ms}ms)"
        )
        return {**result, "success": True, "error": None}
    except Exception as e:
        ms = round((time.time()-t0)*1000)
        console.print(
            f"  ❌ {config['emoji']} [red]{label:10}[/red] "
            f"→ FAILED: {e}  ({ms}ms)"
        )
        return {
            "label": label, "flights": [], "price_insights": {},
            "total_found": 0, "success": False, "error": str(e)
        }


def parallel_flight_search_node(state: TravelState) -> Dict:
    t0      = time.time()
    configs = SEARCH_CONFIGS[:1] if LITE_MODE else SEARCH_CONFIGS

    console.print(
        f"\n[bold blue]🚀 ParallelFlightSearchNode[/bold blue] "
        f"— {len(configs)} search(es) "
        f"{'[dim](lite mode)[/dim]' if LITE_MODE else 'simultaneously'}"
    )

    all_results = []
    with ThreadPoolExecutor(max_workers=len(configs)) as pool:
        futures = {pool.submit(_run_search, cfg, state): cfg for cfg in configs}
        for fut in as_completed(futures, timeout=PROVIDER_TIMEOUT + 5):
            cfg = futures[fut]
            try:
                all_results.append(fut.result(timeout=PROVIDER_TIMEOUT))
            except FTE:
                console.print(f"  ⏰ [yellow]{cfg['label']:10}[/yellow] → TIMEOUT")
                all_results.append({
                    "label": cfg["label"], "flights": [], "price_insights": {},
                    "total_found": 0, "success": False, "error": "timeout"
                })
            except Exception as exc:
                all_results.append({
                    "label": cfg["label"], "flights": [], "price_insights": {},
                    "total_found": 0, "success": False, "error": str(exc)
                })

    ok      = sum(1 for r in all_results if r.get("success"))
    ms      = round((time.time()-t0)*1000)
    seq_est = sum(15000 for _ in configs)

    console.print(
        f"  📊 [bold]{ok}/{len(configs)}[/bold] succeeded | "
        f"parallel={ms}ms vs sequential≈{seq_est}ms "
        f"([green]saved ~{max(0, seq_est-ms):,}ms[/green])"
    )

    insights = next(
        (r["price_insights"] for r in all_results if r.get("success") and r.get("price_insights")),
        {}
    )

    return {
        "flight_results":            all_results,
        "total_searches_attempted":  len(configs),
        "total_searches_succeeded":  ok,
        "price_insights":            insights,
        "execution_trace": [trace_entry(
            "ParallelFlightSearchNode", t0,
            f"{len(configs)} searches (sort=best,cheapest,fastest)",
            f"{ok}/{len(configs)} succeeded in {ms}ms"
        )]
    }


In [14]:
def _dedup_key(f: Dict) -> str:
    """Unique key per physical flight (not sort perspective)."""
    fn = (f.get("flight_number") or "").replace(" ", "")
    if fn:
        return f"{fn}_{f.get('departure_time','')[:10]}"
    return f"{f.get('airline','')}_{f.get('departure_time','')}_{f.get('price_usd','')}"


def _score_flight(f: Dict) -> float:
    """
    Composite score for 'best overall': 60% price, 40% speed.
    Normalises within the pool — lower is better.
    """
    return 0.0


def aggregator_agent(state: TravelState) -> Dict:
    t0      = time.time()
    console.print("\n[bold blue]📊 AggregatorAgent[/bold blue]")

    results = state.get("flight_results", [])
    ok_res  = [r for r in results if r.get("success")]
    fail_res = [r for r in results if not r.get("success")]

    if not ok_res:
        console.print(f"  ❌ All {len(results)} searches failed — no flights available")
        return {
            "best_flight": None, "cheapest_flight": None,
            "fastest_flight": None, "all_flights_ranked": [],
            "execution_trace": [trace_entry("AggregatorAgent", t0,
                f"0/{len(results)} OK", "all failed", "error")]
        }

    if fail_res:
        names = [r["label"] for r in fail_res]
        console.print(f"  ⚠️  Partial failure: {names} — continuing with {len(ok_res)} result set(s)")

    # ── Flatten + deduplicate ────────────────────────────────────────────
    seen, all_flights = {}, []
    for res in ok_res:
        for f in res.get("flights", []):
            k = _dedup_key(f)
            if k not in seen:
                seen[k] = True
                all_flights.append(f)

    if not all_flights:
        console.print("  Searches succeeded but returned 0 flights")
        return {
            "best_flight": None, "cheapest_flight": None,
            "fastest_flight": None, "all_flights_ranked": [],
            "execution_trace": [trace_entry("AggregatorAgent", t0,
                "searches OK", "0 flights", "error")]
        }

    cheapest = min(all_flights, key=lambda x: x["price_usd"])
    fastest  = min(all_flights, key=lambda x: x["duration_min"])

    min_price = min(f["price_usd"]   for f in all_flights)
    max_price = max(f["price_usd"]   for f in all_flights)
    min_dur   = min(f["duration_min"] for f in all_flights)
    max_dur   = max(f["duration_min"] for f in all_flights)
    p_range   = max(max_price - min_price, 1)
    d_range   = max(max_dur   - min_dur,   1)

    def composite(f):
        norm_p = (f["price_usd"]    - min_price) / p_range
        norm_d = (f["duration_min"] - min_dur)   / d_range
        return 0.6 * norm_p + 0.4 * norm_d   # lower = better

    all_flights.sort(key=composite)
    best = all_flights[0]

    ranked = []
    for i, f in enumerate(all_flights[:8]):
        ranked.append({
            "rank":          i + 1,
            "airline":       f["airline"],
            "flight_number": f["flight_number"],
            "price_usd":     f["price_usd"],
            "duration_h":    f["duration_h"],
            "stops":         f["stops_label"],
            "departure":     f["departure_time"],
            "arrival":       f["arrival_time"],
            "cabin":         f["cabin_class"],
            "airplane":      f["airplane"],
            "legroom":       f["legroom"],
        })

    console.print(
        f"  ✅ {len(all_flights)} unique flights | "
        f"cheapest=[green]${cheapest['price_usd']}[/green] {cheapest['airline']} | "
        f"fastest=[green]{fastest['duration_h']}h[/green] {fastest['airline']} | "
        f"best_overall=[green]{best['airline']} ${best['price_usd']}[/green]"
    )

    return {
        "best_flight":       best,
        "cheapest_flight":   cheapest,
        "fastest_flight":    fastest,
        "all_flights_ranked": ranked,
        "execution_trace": [trace_entry("AggregatorAgent", t0,
            f"{len(all_flights)} unique flights from {len(ok_res)} search(es)",
            f"best={best['airline']} ${best['price_usd']}")]
    }

print("✅ AggregatorAgent ready")

✅ AggregatorAgent ready


In [15]:

_FMT_SYSTEM = """You are a friendly, efficient travel assistant.
Format the search results into a clear, engaging response under 150 words.
Structure:
1. Quick summary line (X flights found, date, route)
2. Best overall deal highlighted (airline, price, duration, stops)
3. Cheapest option if different from best
4. Fastest option if different from best
5. One-line alternatives (max 2 more)
6. If any searches failed, briefly note it
7. Price context from insights if available (e.g. "prices are currently low")

Use natural conversational tone. No markdown headers. Use ✈️ 💰 ⚡ emoji sparingly."""

def formatter_agent(state: TravelState) -> Dict:
    t0 = time.time()
    console.print("\n[bold blue]📝 FormatterAgent[/bold blue]")

    if state.get("needs_clarification"):
        msg = state.get("clarification_message", "Could you provide more trip details?")
        console.print("  💬 Clarification needed")
        return {
            "final_response": msg,
            "execution_trace": [trace_entry("FormatterAgent", t0, "clarification", msg[:60])]
        }

    if not state.get("best_flight"):
        msg = ("Sorry, I couldn't find any flights right now. This could be due to "
               "an unavailable route or a temporary issue with the flight search API. "
               "Please try again, or check if the destination and date are correct.")
        console.print("  ❌ No flights found — returning error message")
        return {
            "final_response": msg,
            "execution_trace": [trace_entry("FormatterAgent", t0,
                "no results", msg[:60], "error")]
        }

    p        = state.get("parsed_data") or {}
    insights = state.get("price_insights") or {}

    ctx = {
        "route": {
            "origin":      state.get("origin_display")      or p.get("origin",""),
            "destination": state.get("destination_display") or p.get("destination",""),
            "origin_code": state.get("origin_iata",""),
            "dest_code":   state.get("destination_iata",""),
        },
        "travel_date":       p.get("travel_date",""),
        "passengers":        p.get("num_passengers", 1),
        "cabin_class":       p.get("cabin_class", "economy"),
        "total_flights":     len(state.get("all_flights_ranked") or []),
        "best_overall":      state["best_flight"],
        "cheapest":          state.get("cheapest_flight"),
        "fastest":           state.get("fastest_flight"),
        "top_options":       (state.get("all_flights_ranked") or [])[:5],
        "searches_ok":       state.get("total_searches_succeeded", 0),
        "searches_failed":   (state.get("total_searches_attempted", 0) -
                              state.get("total_searches_succeeded", 0)),
        "price_level":       insights.get("price_level",""),
        "typical_range":     insights.get("typical_range",""),
        "lowest_price_seen": insights.get("lowest_price",""),
    }

    try:
        llm   = get_llm(temperature=0.3)
        chain = ChatPromptTemplate.from_messages([
            ("system", _FMT_SYSTEM),
            ("human", "{ctx}")
        ]) | llm

        resp  = chain.invoke({"ctx": json.dumps(ctx, indent=2)})
        final = resp.content.strip()
        console.print(f"  ✅ Response generated ({len(final)} chars)")
        return {
            "final_response": final,
            "execution_trace": [trace_entry("FormatterAgent", t0,
                f"{ctx['total_flights']} flights", f"{len(final)} chars")]
        }

    except Exception as e:
        b = state["best_flight"]
        c = state.get("cheapest_flight") or b
        f = state.get("fastest_flight")  or b
        fallback = (
            f"✈️  Found {ctx['total_flights']} flights "
            f"from {ctx['route']['origin']} to {ctx['route']['destination']} "
            f"on {ctx['travel_date']}.\n\n"
            f"🏆 Best overall:  {b['airline']} "
            f"${b['price_usd']} | {b['duration_h']}h | {b['stops_label']} | "
            f"Departs {b['departure_time']}\n"
            f"💰 Cheapest:      {c['airline']} ${c['price_usd']}\n"
            f"⚡ Fastest:       {f['airline']} {f['duration_h']}h\n"
        )
        if ctx["searches_failed"]:
            fallback += f"\n⚠️  {ctx['searches_failed']} search perspective(s) unavailable."
        console.print(f"  ⚡ Fallback template (LLM error: {e})")
        return {
            "final_response": fallback,
            "execution_trace": [trace_entry("FormatterAgent", t0,
                "LLM fallback", "template", "fallback")]
        }

print("✅ FormatterAgent ready")

✅ FormatterAgent ready


In [16]:

from IPython.display import display, Markdown

def _route_after_validate(state: TravelState) -> str:
    return "format" if state.get("needs_clarification") else "search_flights"

def build_graph():
    g = StateGraph(TravelState)

    g.add_node("parse",          parser_agent)
    g.add_node("resolve_airports", airport_resolver_agent)
    g.add_node("validate",       validator_agent)
    g.add_node("search_flights", parallel_flight_search_node)
    g.add_node("aggregate",      aggregator_agent)
    g.add_node("format",         formatter_agent)

    g.add_edge(START,              "parse")
    g.add_edge("parse",            "resolve_airports")
    g.add_edge("resolve_airports", "validate")

    g.add_conditional_edges("validate", _route_after_validate, {
        "search_flights": "search_flights",
        "format":         "format",
    })

    g.add_edge("search_flights", "aggregate")
    g.add_edge("aggregate",      "format")
    g.add_edge("format",         END)

    return g.compile()

app = build_graph()

mermaid = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	resolve_airports(resolve_airports)
	validate(validate)
	search_flights(search_flights)
	aggregate(aggregate)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	aggregate --> format;
	parse --> resolve_airports;
	resolve_airports --> validate;
	search_flights --> aggregate;
	validate -.-> format;
	validate -.-> search_flights;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [17]:

STATUS_COLOR = {
    "success":             "green",
    "error":               "red",
    "fallback":            "yellow",
    "clarification_needed":"yellow",
    "skipped":             "dim",
}

def run(query: str, force_fail: set = None, history: List = None):
    global FORCE_FAIL_SEARCHES
    FORCE_FAIL_SEARCHES = force_fail or set()

    console.print(Panel(
        f"[bold white]{query}[/bold white]",
        title="🛫  Travel Request", border_style="blue"
    ))

    state   = make_state(query, history)
    t_start = time.time()
    final   = app.invoke(state)
    total   = round((time.time() - t_start) * 1000)

    tbl = Table(title="⚡ Execution Trace", box=box.ROUNDED,
                show_header=True, header_style="bold magenta", width=100)
    tbl.add_column("Agent",  style="cyan", width=26)
    tbl.add_column("Status", width=22)
    tbl.add_column("Time",   width=8)
    tbl.add_column("Output", width=44)

    for e in final.get("execution_trace", []):
        c = STATUS_COLOR.get(e.get("status",""), "white")
        tbl.add_row(
            e["agent"],
            f"[{c}]{e.get('status','')}[/{c}]",
            f"{e.get('duration_ms',0):.0f}ms",
            (e.get("output_summary","") or "")[:44],
        )

    console.print(tbl)

    ok  = final.get("total_searches_succeeded", 0)
    tot = final.get("total_searches_attempted", 0)
    insights = final.get("price_insights") or {}
    insight_note = ""
    if insights.get("price_level"):
        insight_note = f" | price level: {insights['price_level']}"
    if insights.get("typical_range"):
        insight_note += f" | typical range: ${insights['typical_range'][0]}-${insights['typical_range'][1]}"

    console.print(
        f"[dim]Pipeline total: {total}ms | "
        f"Searches: {ok}/{tot} OK{insight_note}[/dim]"
    )

    console.print(Panel(
        final.get("final_response", "—"),
        title="🤖  Assistant", border_style="green"
    ))

    return final


In [18]:
from rich.rule import Rule

console.print(Rule("[bold yellow]TEST 1 — Happy Path (all 3 searches succeed)"))
s1 = run("Book me a flight from Kolkata to Dibrugarh next Friday")

────────────────────────────────── TEST 1 — Happy Path (all 3 searches succeed) ───────────────────────────────────

╭────────────────────────────────────────────── 🛫  Travel Request ───────────────────────────────────────────────╮
│ Book me a flight from Kolkata to Dibrugarh next Friday                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔤 ParserAgent

✅ dest=Dibrugarh origin=Kolkata date=2026-04-03 pax=1 cabin=economy

🗺️  AirportResolverAgent

...

📝 FormatterAgent

✅ Response generated (407 chars)

                                         ⚡ Execution Trace                                         
╭─────────────────────────┬─────────────────────┬──────┬───────────────────────────────────────────╮
│ Agent                   │ Status              │ Time │ Output                                    │
├─────────────────────────┼─────────────────────┼──────┼───────────────────────────────────────────┤
│ ParserAgent             │ success             │ 707… │ dest=Dibrugarh, date=2026-04-03           │
│ AirportResolverAgent    │ success             │ 375… │ CCU + DIB                                 │
│ ValidatorAgent          │ success             │ 4ms  │ valid=True, missing=[]                    │
│ ParallelFlightSearchNo… │ success             │ 554… │ 3/3 succeeded in 5540ms                   │
│ AggregatorAgent         │ success             │ 4ms  │ best=IndiGo $157                          │
│ FormatterAgent          │ success             │ 108… │ 407 chars                                 │
╰─────────────────────────┴─────────────────────┴──────┴───────────────────────────────────────────╯

Pipeline total: 7734ms | Searches: 3/3 OK | price level: high

╭───────────────────────────────────────────────── 🤖  Assistant ─────────────────────────────────────────────────╮
│ 5 flights found from Kolkata (CCU) to Dibrugarh (DIB) on April 3, 2026. The best overall deal is with IndiGo    │
│ for $157, taking 6.8 hours with 1 stop. The cheapest option is also with IndiGo for $157. If you're in a hurry, │
│ consider IndiGo's direct flight for $175, taking just 1.7 hours ⚡. Alternative options include Air India       │
│ flights. All searches were successful, but note that prices are currently high 💰.                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
console.print(Rule("[bold yellow]TEST 2 — Partial Failure (cheapest search fails)"))
s2 = run(
    "Find flights to Tokyo for December 10th",
    force_fail={"cheapest"}   # only cheapest search fails; best + fastest still run
)

──────────────────────────────── TEST 2 — Partial Failure (cheapest search fails) ─────────────────────────────────

╭────────────────────────────────────────────── 🛫  Travel Request ───────────────────────────────────────────────╮
│ Find flights to Tokyo for December 10th                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔤 ParserAgent

✅ dest=Tokyo origin=New York date=2026-12-10 pax=1 cabin=economy

🗺️  AirportResolverAgent

...

📝 FormatterAgent

✅ Response generated (428 chars)

                                         ⚡ Execution Trace                                         
╭─────────────────────────┬─────────────────────┬──────┬───────────────────────────────────────────╮
│ Agent                   │ Status              │ Time │ Output                                    │
├─────────────────────────┼─────────────────────┼──────┼───────────────────────────────────────────┤
│ ParserAgent             │ success             │ 553… │ dest=Tokyo, date=2026-12-10               │
│ AirportResolverAgent    │ success             │ 488… │ JFK + NRT                                 │
│ ValidatorAgent          │ success             │ 3ms  │ valid=True, missing=[]                    │
│ ParallelFlightSearchNo… │ success             │ 155… │ 2/3 succeeded in 1553ms                   │
│ AggregatorAgent         │ success             │ 6ms  │ best=Air Canada $918                      │
│ FormatterAgent          │ success             │ 929… │ 428 chars                                 │
╰─────────────────────────┴─────────────────────┴──────┴───────────────────────────────────────────╯

Pipeline total: 3539ms | Searches: 2/3 OK | price level: typical

╭───────────────────────────────────────────────── 🤖  Assistant ─────────────────────────────────────────────────╮
│ 8 flights found from New York (JFK) to Tokyo (NRT) on December 10, 2026. The best overall deal is with Air      │
│ Canada for $918, a 16.8-hour flight with 1 stop. This is also the cheapest option. The fastest option is also   │
│ with Air Canada, with the same details. Other alternatives include Alaska for $1022 and Korean Air for $1047.   │
│ One search failed. Prices are currently at a typical level, with $918 being the lowest price seen. ✈️            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
console.print(Rule("[bold yellow]TEST 4 — Missing Info → Clarification"))
s4 = run("I want to travel somewhere next month")

────────────────────────────────────── TEST 4 — Missing Info → Clarification ──────────────────────────────────────

╭────────────────────────────────────────────── 🛫  Travel Request ───────────────────────────────────────────────╮
│ I want to travel somewhere next month                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔤 ParserAgent

✅ dest=None origin=New York date=2026-05-02 pax=1 cabin=economy

🗺️  AirportResolverAgent

...

📝 FormatterAgent

💬 Clarification needed

                                         ⚡ Execution Trace                                         
╭─────────────────────────┬─────────────────────┬──────┬───────────────────────────────────────────╮
│ Agent                   │ Status              │ Time │ Output                                    │
├─────────────────────────┼─────────────────────┼──────┼───────────────────────────────────────────┤
│ ParserAgent             │ success             │ 539… │ dest=None, date=2026-05-02                │
│ AirportResolverAgent    │ skipped             │ 2ms  │ skipped                                   │
│ ValidatorAgent          │ clarification_need… │ 3ms  │ valid=False, missing=['destination']      │
│ FormatterAgent          │ success             │ 3ms  │ I need a bit more info: Where would you   │
│                         │                     │      │ like                                      │
╰─────────────────────────┴─────────────────────┴──────┴───────────────────────────────────────────╯

Pipeline total: 551ms | Searches: 0/0 OK

╭───────────────────────────────────────────────── 🤖  Assistant ─────────────────────────────────────────────────╮
│ I need a bit more info: Where would you like to travel? (city name)                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
console.print(Rule("[bold yellow]TEST 5 — Multi-passenger Business Class"))
s5 = run("2 business class tickets to Dubai on December 20th")

───────────────────────────────────── TEST 5 — Multi-passenger Business Class ─────────────────────────────────────

╭────────────────────────────────────────────── 🛫  Travel Request ───────────────────────────────────────────────╮
│ 2 business class tickets to Dubai on December 20th                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔤 ParserAgent

✅ dest=Dubai origin=New York date=2026-12-20 pax=2 cabin=business

🗺️  AirportResolverAgent

...

📝 FormatterAgent

✅ Response generated (461 chars)

                                         ⚡ Execution Trace                                         
╭─────────────────────────┬─────────────────────┬──────┬───────────────────────────────────────────╮
│ Agent                   │ Status              │ Time │ Output                                    │
├─────────────────────────┼─────────────────────┼──────┼───────────────────────────────────────────┤
│ ParserAgent             │ success             │ 550… │ dest=Dubai, date=2026-12-20               │
│ AirportResolverAgent    │ success             │ 438… │ JFK + DXB                                 │
│ ValidatorAgent          │ success             │ 3ms  │ valid=True, missing=[]                    │
│ ParallelFlightSearchNo… │ success             │ 373… │ 3/3 succeeded in 3736ms                   │
│ AggregatorAgent         │ success             │ 4ms  │ best=EgyptAir $6901                       │
│ FormatterAgent          │ success             │ 103… │ 461 chars                                 │
╰─────────────────────────┴─────────────────────┴──────┴───────────────────────────────────────────╯

Pipeline total: 5774ms | Searches: 3/3 OK | price level: typical

╭───────────────────────────────────────────────── 🤖  Assistant ─────────────────────────────────────────────────╮
│ 8 flights found from New York (JFK) to Dubai (DXB) on December 20, 2026. The best overall deal is with EgyptAir │
│ for $6,901, a 17.2-hour journey with 1 stop. The cheapest option is Royal Air Maroc for $4,989, but it takes    │
│ 32.2 hours with 1 stop. For a faster trip, consider Emirates for $13,446, a direct 12.4-hour flight ✈️. Other    │
│ options include American and Turkish Airlines. Prices are currently at a typical level, with the lowest price   │
│ seen being $4,989 💰.                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯